# 24-Hour Flow Model Training

Runs `experiments/24hr_flow_study.json` and `experiments/24hr_flow_study_noSTG.json`
through the ML pipeline via `run_experiments.py`.

**Data:** `data/Merged/Miami_GWL_WL_RAIN_GATE_FLOW_2017_2024.csv`  
**Train:** 2017–2023 &nbsp;|&nbsp; **Test:** 2024  
**Target:** `gwl` at +24 h lead time  
**Models:** LR, RF, MLP  
**Results:** `results/<experiment_name>/test/results.csv`

---

## Lag Range Reference

Each input column is represented as a window of lagged (and potentially future) values. The lag range `[a, b]` means features are constructed from hour `a` to hour `b` relative to the prediction time:

- **`[-24, 0]` — historical only:** the model sees observations from 24 h ago up to the current timestep. No future information is provided.
- **`[-24, 24]` — perfect prognostic:** the model also receives values up to 24 h into the future (i.e. out to the full lead time). This simulates a perfect forecast and represents an upper bound on what that variable could contribute if a reliable forecast were available.

---

## Experiment Configurations — `24hr_flow_study.json`

### `24hr_flow_all_inputs`
All inputs available at perfect-prog resolution. Establishes the ceiling for flow-augmented performance.

| Column | Lag Range | Mode |
|--------|-----------|------|
| gwl | [-24, 0] | Historical only |
| wl | [-24, 24] | Perfect-prog |
| rain | [-24, 24] | Perfect-prog |
| stgH | [-24, 24] | Perfect-prog |
| stgT | [-24, 24] | Perfect-prog |
| gate1 | [-24, 24] | Perfect-prog |
| gate2 | [-24, 24] | Perfect-prog |
| flow | [-24, 24] | Perfect-prog |

### `24hr_flow_gwl_wl_flow`
Isolates the combined predictive value of water level and flow forecasts, with all other variables restricted to historical observations.

| Column | Lag Range | Mode |
|--------|-----------|------|
| gwl | [-24, 0] | Historical only |
| wl | [-24, 24] | Perfect-prog |
| rain | [-24, 0] | Historical only |
| stgH | [-24, 0] | Historical only |
| stgT | [-24, 0] | Historical only |
| gate1 | [-24, 0] | Historical only |
| gate2 | [-24, 0] | Historical only |
| flow | [-24, 24] | Perfect-prog |

### `24hr_flow_gwl_flow`
Only flow is given perfect-prog knowledge; everything else is historical. Tests whether flow alone can drive 24 h GWL prediction.

| Column | Lag Range | Mode |
|--------|-----------|------|
| gwl | [-24, 0] | Historical only |
| wl | [-24, 0] | Historical only |
| rain | [-24, 0] | Historical only |
| stgH | [-24, 0] | Historical only |
| stgT | [-24, 0] | Historical only |
| gate1 | [-24, 0] | Historical only |
| gate2 | [-24, 0] | Historical only |
| flow | [-24, 24] | Perfect-prog |

### `24hr_flow_gwl_rain_gates`
Flow paired with rain and gate forecasts, but not water level. Tests the flow + forcing variable combination without WL.

| Column | Lag Range | Mode |
|--------|-----------|------|
| gwl | [-24, 0] | Historical only |
| rain | [-24, 24] | Perfect-prog |
| stgH | [-24, 0] | Historical only |
| stgT | [-24, 0] | Historical only |
| gate1 | [-24, 24] | Perfect-prog |
| gate2 | [-24, 24] | Perfect-prog |
| flow | [-24, 24] | Perfect-prog |

---

## Experiment Configurations — `24hr_flow_study_noSTG.json`

### `24hr_flow_all_inputs_noSTG`
Identical to `24hr_flow_all_inputs` except `stgH` and `stgT` are restricted to historical observations (`[-24, 0]`). Directly quantifies the marginal contribution of perfect-prognosis stage data.

| Column | Lag Range | Mode |
|--------|-----------|------|
| gwl | [-24, 0] | Historical only |
| wl | [-24, 24] | Perfect-prog |
| rain | [-24, 24] | Perfect-prog |
| **stgH** | **[-24, 0]** | **Historical only** |
| **stgT** | **[-24, 0]** | **Historical only** |
| gate1 | [-24, 24] | Perfect-prog |
| gate2 | [-24, 24] | Perfect-prog |
| flow | [-24, 24] | Perfect-prog |

Import utilities for JSON parsing, subprocess execution, and path resolution. Resolve the project root and all experiment file paths relative to this notebook's location in `notebooks/`.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path("../").resolve()  # project root from notebooks/
EXPERIMENT_FILES = [
    ROOT / "experiments" / "24hr_flow_study.json",
    ROOT / "experiments" / "24hr_flow_study_noSTG.json",
]

## Experiment Configuration

Load all experiment JSON files and print each experiment's name, input columns, and model choices — a quick sanity check before kicking off training.

In [ ]:
for experiment_file in EXPERIMENT_FILES:
    with open(experiment_file) as f:
        config = json.load(f)

    print(f"── {experiment_file.name} ──")
    for exp in config["experiments"]:
        print(f"  {exp['experiment_name']}")
        print(f"    inputs: {[s['column'] for s in exp['input_specifications']]}")
        print(f"    models: {exp['model_architectures']}")
        print()
    print()

## Run Training

Launch `run_experiments.py` as a subprocess for each experiment file, with `cwd` set to the project root so all internal relative paths resolve correctly. Training output streams live to this cell. Raises an error if the script exits with a non-zero code. Already-completed experiments are skipped automatically.

In [ ]:
for experiment_file in EXPERIMENT_FILES:
    print(f"\n{'='*60}")
    print(f"Running {experiment_file.name} ...")
    print(f"{'='*60}")
    result = subprocess.run(
        [
            sys.executable, "run_experiments.py",
            "-e", str(experiment_file.relative_to(ROOT)),
        ],
        cwd=ROOT,
        capture_output=False,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"run_experiments.py exited with code {result.returncode} "
            f"for {experiment_file.name}"
        )

## Results

Read each experiment's `test/results.csv`, prepend the experiment name as a column, and concatenate into a single summary DataFrame for comparison across all experiments and three models.

In [ ]:
import pandas as pd

all_rows = []
for experiment_file in EXPERIMENT_FILES:
    with open(experiment_file) as f:
        config = json.load(f)

    for exp in config["experiments"]:
        path = ROOT / "results" / exp["experiment_name"] / "test" / "results.csv"
        if path.exists():
            df = pd.read_csv(path, index_col=0)
            df.insert(0, "experiment", exp["experiment_name"])
            all_rows.append(df)
        else:
            print(f"WARNING: not found — {path}")

pd.concat(all_rows, ignore_index=True)